# Кластеризация на разных экспериментах разных пайплайнов

## 1. Базовые функции

In [1]:
import sys
from dotenv import load_dotenv

sys.path.append('./../')
load_dotenv()

True

In [2]:
import os
from typing import Any, Dict, List, TypedDict, Tuple
import json
import numpy as np
import pandas as pd

In [3]:
QUERIES_CNT = 16
RUNS = 8

# Функции генерации схем

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
from eval.clustering import compute_similarity_stats, compute_auc_separation
from eval.md_classes import MDField, DataStep

def get_fields_by_block(df: pd.DataFrame, q_id: int, r_id: int) -> Dict[str, Dict[str, MDField]]:
    fields_by_block = {}
    fields_ids_by_block = {}

    loc_df = df.loc[(df['query_id'] == q_id) & (df['run_id'] == r_id)]
    for r_id, row in loc_df.iterrows():
        block_id = row['node_id']
        block_fields = fields_by_block.get(block_id, {})
        fields_ids = fields_ids_by_block.get(block_id, {})

        block_info = DataStep(row['node_id'], row['node_descr'])
        md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block_info)
        
        block_fields[row['field_name']] = md_field
        fields_ids[row['field_name']] = r_id

        fields_by_block[block_id] = block_fields
        fields_ids_by_block[block_id] = fields_ids

    return fields_by_block, fields_ids_by_block

def gen_openai_embed_func(full_emb_path: str, short_emb_path: str):
    def get_embeds(fields_ids: List[int], needs_full_emb: bool):
        embeds = []
        for fid in fields_ids:
            if needs_full_emb:
                file_path = os.path.join(full_emb_path, f'{fid}.npy')
            else:
                file_path = os.path.join(short_emb_path, f'{fid}.npy')
            emb = np.load(file_path)
            embeds.append(emb)
        return np.array(embeds)
    return get_embeds

def gen_hf_ebedding_func(model_name: str, fields_info_df: pd.DataFrame):
    model = SentenceTransformer(model_name)

    def get_embeds(fields_ids: List[int], needs_full_emb: bool):
        strings = []
        for fid in fields_ids:
            row = fields_info_df.iloc[fid]            
            if needs_full_emb:
                block_info = DataStep(row['node_id'], row['node_descr']) 
            else:
                block_info = None
            md_field = MDField(row['field_name'], row['field_description'], row['field_type'], block=block_info)
            new_str = str(md_field)
            strings.append(new_str)

        embeds = model.encode(strings,  normalize_embeddings=True, batch_size=64, show_progress_bar=False)
        return embeds
    
    return get_embeds

/home/alex/Documents/GitHub/mas_workflow/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Генерация статистики по M4 данным для разных эмбеддингов

In [5]:
df = pd.read_csv('./../data/openai/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv', index_col=[0])
len(df)

28999

In [6]:
MODELS = [
    'pritamdeka/S-BioBert-snli-multinli-stsb',
    "all-MiniLM-L6-v2",
    #"BAAI/bge-m3",
    "Qwen/Qwen3-Embedding-0.6B",
    "OpenAI"
]

EMBEDS = {
    "OpenAI": gen_openai_embed_func('./../data/openai/md_generations/embeds/m4_full', './../data/openai/md_generations/embeds/m4_fields_only'),
    "all-MiniLM-L6-v2": gen_hf_ebedding_func("all-MiniLM-L6-v2", df),
    #"BAAI/bge-m3": gen_hf_ebedding_func("BAAI/bge-m3", df),
    "Qwen/Qwen3-Embedding-0.6B": gen_hf_ebedding_func("Qwen/Qwen3-Embedding-0.6B", df),
    'pritamdeka/S-BioBert-snli-multinli-stsb': gen_hf_ebedding_func("pritamdeka/S-BioBert-snli-multinli-stsb", df)
}

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23383.94it/s]


In [7]:
from scipy.stats import wilcoxon

def count_avg_statistic(df: pd.DataFrame, queries: int, runs: int, emb_func):
    no_block_stats = []
    dag_block_stats = []
    
    for q in range(queries):
        for r in range(runs):
            _, fields_ids = get_fields_by_block(df, q, r)

            flat_blocks_keys = []
            flat_blocks_ids = []
            for bl_idx, block_name in enumerate(fields_ids.keys()):
                block_fields = fields_ids[block_name]
                for _, fld_idx in block_fields.items():
                    flat_blocks_keys.append(bl_idx)
                    flat_blocks_ids.append(fld_idx)

            short_embeds = emb_func(flat_blocks_ids, needs_full_emb=False)
            full_embeds = emb_func(flat_blocks_ids, needs_full_emb=True)

            new_short_stat = compute_similarity_stats(short_embeds, flat_blocks_keys)
            new_full_stat = compute_similarity_stats(full_embeds, flat_blocks_keys)

            short_aug_score = compute_auc_separation(short_embeds, flat_blocks_keys)
            full_aug_score = compute_auc_separation(full_embeds, flat_blocks_keys)


            no_block_stats.append({
                "within_block_mean": new_short_stat.within_block_mean,
                "within_block_std": new_short_stat.within_block_std,
                "cross_block_mean": new_short_stat.cross_block_mean,
                "cross_block_std": new_short_stat.cross_block_std,
                "separation": new_short_stat.separation,
                "aug_score": short_aug_score,
            })
            dag_block_stats.append({
                "within_block_mean": new_full_stat.within_block_mean,
                "within_block_std": new_full_stat.within_block_std,
                "cross_block_mean": new_full_stat.cross_block_mean,
                "cross_block_std": new_full_stat.cross_block_std,
                "separation": new_full_stat.separation,
                "aug_score": full_aug_score,
            })

    return no_block_stats, dag_block_stats


In [8]:
def rank_biserial_wilcoxon(stat, n):
    max_stat = n * (n + 1) / 2
    return (2 * stat) / max_stat - 1

for m in MODELS:
    emb_func = EMBEDS[m]
    no_block_stats, dag_block_stats = count_avg_statistic(df, QUERIES_CNT, RUNS, emb_func)
    no_block_stats_df = pd.DataFrame(no_block_stats)
    dag_block_stats_df = pd.DataFrame(dag_block_stats)
    
    print(f'---{m} STATS ---')
    
    print('1) No dag block stats')
    print(no_block_stats_df.describe())
    print('\n\n2) With DAG blocks stats')
    print(dag_block_stats_df.describe())

    print('\n\n3) Wilcoxon signed-rank test on separation comparing:')
    sep_short = np.array([some_stat["separation"] for some_stat in no_block_stats])
    sep_full = np.array([some_stat["separation"] for some_stat in dag_block_stats])
    
    stat_short, p_short = wilcoxon(sep_short, alternative='greater')
    r_short = rank_biserial_wilcoxon(stat_short, sep_short.shape[0])

    # Test augmented: is separation > 0?
    stat_full, p_full = wilcoxon(sep_full, alternative='greater')
    r_full = rank_biserial_wilcoxon(stat_full, sep_short.shape[0])

    print(f"Flat: W={stat_short:.0f}, p={p_short:.4e}, r={r_short:.3f}, "
        f"mean={sep_short.mean():.4f}±{sep_short.std():.4f}")
    print(f"DAG augm:  W={stat_full:.0f},  p={p_full:.4e},  r={r_full:.3f},  "
        f"mean={sep_full.mean():.4f}±{sep_full.std():.4f}")
    
    print('\n\n4) Wilcoxon signed-rank test on separation increasing:')
    stat, p = wilcoxon(sep_full, sep_short, alternative='greater')
    print(f"Wilcoxon signed-rank: W={stat:.0f}, p={p:.4e}")
    print(f"Delta-sep stat: {np.mean(sep_full - sep_short)}+-{np.std(sep_full - sep_short)} (also delta means: {np.mean(sep_full) - np.mean(sep_short)})")
    r = rank_biserial_wilcoxon(stat, len(sep_short))
    print(f"R effect size: {r}")

    print('\n\n5) ROC-auc metric improvement')
    auc_short = np.array([some_stat["aug_score"] for some_stat in no_block_stats])
    auc_full = np.array([some_stat["aug_score"] for some_stat in dag_block_stats])
    stat, p = wilcoxon(auc_full, auc_short, alternative='greater')
    r=rank_biserial_wilcoxon(stat, auc_short.shape[0])
    print(f"Flat AUC:  {np.mean(auc_short):.3f} ± {np.std(auc_short):.3f}")
    print(f"Aug  AUC:  {np.mean(auc_full):.3f} ± {np.std(auc_full):.3f}")
    print(f"Wilcoxon:  W={stat:.0f}, p={p:.4e}, r={r:.3f}")
    


---pritamdeka/S-BioBert-snli-multinli-stsb STATS ---
1) No dag block stats
       within_block_mean  within_block_std  cross_block_mean  cross_block_std  \
count         128.000000        128.000000        128.000000       128.000000   
mean            0.660161          0.102014          0.496274         0.109952   
std             0.026574          0.008224          0.021351         0.006639   
min             0.605004          0.077603          0.441248         0.095563   
25%             0.639113          0.096441          0.479839         0.104990   
50%             0.658974          0.101718          0.495498         0.109436   
75%             0.677245          0.107046          0.508880         0.114370   
max             0.731417          0.124124          0.560154         0.127071   

       separation   aug_score  
count  128.000000  128.000000  
mean     0.163886    0.861447  
std      0.016077    0.027161  
min      0.123545    0.782100  
25%      0.153014    0.842974  
50%

## Статистика ВСЕХ DAG генераций

In [ ]:
from scatter_blocks import plot_fields_scatter, plot_comparison

md_paths = {
    'm3': './../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv',
    'm4': './../data/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv',
}

embeds_short = {
    'm3': './../data/md_generations/embeds/m3_fields_only',
    'm4': './../data/md_generations/embeds/m4_fields_only',
}

embeds_full = {
    'm3': './../data/md_generations/embeds/m3_full',
    'm4': './../data/md_generations/embeds/m4_full',
}

SAVE_DIR = './compare_res'

for p_name, p_path in md_paths.items():
    df = pd.read_csv(p_path, index_col=[0])
    emb_func = gen_embed_func(embeds_full[p_name], embeds_short[p_name])

    for q_id in range(QUERIES_CNT):
        for run in range(RUNS):
            #if run != 3:
            #    continue

            total_path = os.path.join(SAVE_DIR, p_name, f"{q_id}_{run}")
            os.makedirs(total_path, exist_ok=True)

            fields_per_block, fields_ids = get_fields_by_block(df, q_id, run)
            plot_comparison(
                md_fields_per_block = fields_per_block,
                md_fields_ids = fields_ids,
                embed_fn = emb_func,
                save_html = os.path.join(total_path, 'comp_plot.html'),
            )

In [5]:
from scatter_blocks import plot_fields_scatter, plot_comparison

md_paths = {
#    'm3': './../data/md_generations/m3_synth_bio_mds/m3_synth_bio_mds.csv',
    'm4': './../data/openai/md_generations/m4_synth_bio_mds/m4_synth_bio_mds.csv',
}

embeds_short = {
#    'm3': './../data/md_generations/embeds/m3_fields_only',
    'm4': './../data/openai/md_generations/embeds/m4_fields_only',
}

embeds_full = {
#    'm3': './../data/md_generations/embeds/m3_full',
    'm4': './../data/openai/md_generations/embeds/m4_full',
}

SAVE_DIR = './tmp_paper_res'

for p_name, p_path in md_paths.items():
    df = pd.read_csv(p_path, index_col=[0])
    emb_func = gen_openai_embed_func(embeds_full[p_name], embeds_short[p_name])

    for q_id in [0]:
        for run in [3]:
            #if run != 3:
            #    continue

            total_path = os.path.join(SAVE_DIR, p_name, f"{q_id}_{run}")
            os.makedirs(total_path, exist_ok=True)

            fields_per_block, fields_ids = get_fields_by_block(df, q_id, run)
            plot_comparison(
                md_fields_per_block = fields_per_block,
                md_fields_ids = fields_ids,
                embed_fn = emb_func,
                save_html = os.path.join(total_path, 'comp_plot.html'),
                save_png = os.path.join(total_path, 'comp_plot.png'),
                png_dpi_scale=4,
                legend_font_size = 17,
                axis_font_size = 24,
                tick_font_size = 22,
                subplot_title_font_size = 28,
                point_size = 14,
                drawing_ellipses_names = [
                    'raw_visium_reads', 
                    'raw_sc_rna_reads', 
                    'raw_cite_reads', 
                    'visium_section_prep',
                    't_cell_exhaustion_states',
                    'survival_outcome_record',
                    'ca19_9_record',
                    'clinical_response_record',],
            )

I0000 00:00:1779748469.242411   12642 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/home/alex/Documents/GitHub/mas_workflow/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Dimensionality reduction: UMAP (3072D → 2D, 214 points)


/home/alex/Documents/GitHub/mas_workflow/.venv/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


  Dimensionality reduction: UMAP (3072D → 2D, 214 points)
Saved HTML → ./tmp_paper_res/m4/0_3/comp_plot.html
Saved PNG → ./tmp_paper_res/m4/0_3/comp_plot.png (~384 DPI)
